# Task 2: Predict Future Stock Prices (Short-Term)
**DevelopersHub Corp — AI/ML Internship**

---

## Problem Statement

Stock price prediction is a classic regression problem in financial machine learning.
The goal is to predict the **next day's closing price** of Apple Inc. (AAPL) stock
using historical OHLCV data and engineered time-series features.

We train and compare two models:
- **Linear Regression** — simple, interpretable baseline
- **Random Forest Regressor** — ensemble model that captures non-linear patterns

## Dataset

| Property | Detail |
|---|---|
| **Stock** | Apple Inc. (AAPL) |
| **Source** | Yahoo Finance via `yfinance` library |
| **Period** | January 2022 – December 2024 |
| **Rows** | ~782 trading days |
| **Raw Features** | Open, High, Low, Close, Volume |
| **Target** | Next day's Close price |

---

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import yfinance as yf

plt.rcParams['figure.dpi'] = 110
print('✅ Libraries imported')

## Load Stock Data via yfinance

In [ ]:
# Download Apple stock data — 3 years of daily OHLCV data
ticker = 'AAPL'
df_raw = yf.download(ticker, start='2022-01-01', end='2024-12-31',
                     auto_adjust=True, progress=False)

# Flatten multi-level columns if present
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

df_raw = df_raw[['Open','High','Low','Close','Volume']]
print(f'Shape : {df_raw.shape}')
print(f'Period: {df_raw.index[0].date()} → {df_raw.index[-1].date()}')
df_raw.head()

### Quick look at the data

In [ ]:
print('\n--- Info ---')
df_raw.info()
print('\n--- Descriptive Stats ---')
df_raw.describe().round(2)

## Feature Engineering

> **Important:** We only use *previous day* data as features.
> Using same-day Open/High/Low would be data leakage — in real life,
> you don't know tomorrow's Open when making a prediction tonight.

Features created:
- Previous day OHLCV values
- High-Low range and Open-Close difference (price action signals)
- Rolling Moving Averages: MA5, MA10, MA20
- Momentum (3-day price change)
- Rolling volatility (5-day std of close)

In [ ]:
df = df_raw.copy()

# Target: tomorrow's closing price
df['Target'] = df['Close']

# Previous day features (shift by 1 to avoid leakage)
df['Prev_Close']    = df['Close'].shift(1)
df['Prev_Open']     = df['Open'].shift(1)
df['Prev_High']     = df['High'].shift(1)
df['Prev_Low']      = df['Low'].shift(1)
df['Prev_Volume']   = df['Volume'].shift(1)
df['Prev_HL_Range'] = (df['High'] - df['Low']).shift(1)
df['Prev_OC_Diff']  = (df['Open'] - df['Close']).shift(1)

# Rolling statistics (computed on previous days)
df['MA_5']          = df['Close'].shift(1).rolling(5).mean()
df['MA_10']         = df['Close'].shift(1).rolling(10).mean()
df['MA_20']         = df['Close'].shift(1).rolling(20).mean()
df['Momentum_3']    = df['Close'].shift(1) - df['Close'].shift(4)
df['Volatility_5']  = df['Close'].shift(1).rolling(5).std()

df.dropna(inplace=True)

FEATURES = ['Prev_Close','Prev_Open','Prev_High','Prev_Low','Prev_Volume',
            'Prev_HL_Range','Prev_OC_Diff','MA_5','MA_10','MA_20',
            'Momentum_3','Volatility_5']
TARGET = 'Target'

print(f'Features : {len(FEATURES)}')
print(f'Rows after dropna: {len(df)}')
df[FEATURES + [TARGET]].head()

## Train / Test Split

> We use a **time-based 80/20 split** — never shuffle time series data!
> Shuffling would leak future information into training.

In [ ]:
X = df[FEATURES].values
y = df[TARGET].values
dates = df.index

split = int(len(X) * 0.80)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
dates_train, dates_test = dates[:split], dates[split:]

print(f'Training samples : {len(X_train)}  ({dates_train[0].date()} → {dates_train[-1].date()})')
print(f'Test samples     : {len(X_test)}  ({dates_test[0].date()}  → {dates_test[-1].date()})')

# Scale features (important for Linear Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit only on train
X_test_sc  = scaler.transform(X_test)         # transform test with same scaler
print('\nFeature scaling done ✅')

## Model Training

### Model 1 — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
print('Linear Regression trained ✅')

### Model 2 — Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,   # 300 decision trees
    max_depth=8,        # limit depth to prevent overfitting
    random_state=42,
    n_jobs=-1           # use all CPU cores
)
rf.fit(X_train_sc, y_train)
rf_pred = rf.predict(X_test_sc)
print('Random Forest trained ✅')

## Model Evaluation

Three metrics used:
| Metric | What it measures |
|---|---|
| **RMSE** | Root Mean Squared Error — penalizes large errors more |
| **MAE** | Mean Absolute Error — average dollar error per prediction |
| **R²** | Coefficient of determination — 1.0 = perfect, 0 = baseline |

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f'{name}')
    print(f'  RMSE : ${rmse:.4f}')
    print(f'  MAE  : ${mae:.4f}')
    print(f'  R²   : {r2:.4f}')
    print()
    return rmse, mae, r2

print('=' * 45)
print('  EVALUATION RESULTS ON TEST SET')
print('=' * 45)
lr_m = evaluate('Linear Regression', y_test, lr_pred)
rf_m = evaluate('Random Forest',     y_test, rf_pred)

## Visualizations

### Plot 1 — Actual vs Predicted Close Price

In [ ]:
BG='#0F1117'; CARD='#1A1D27'; TEXT='#E8E8F0'; MUTED='#6B7280'
C_ACT='#4ECDC4'; C_LR='#FF6B6B'; C_RF='#F5D547'; C_GRID='#2A2D3A'

def style(ax):
    ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_color(C_GRID)
    ax.tick_params(colors=MUTED)
    ax.grid(color=C_GRID, linewidth=0.5)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
fig.patch.set_facecolor(BG)
fig.suptitle('AAPL — Actual vs Predicted Next-Day Close Price',
             color=TEXT, fontsize=16, fontweight='bold', y=1.01)

for ax, title, pred, col, m in zip(
    axes,
    ['Linear Regression', 'Random Forest (300 trees)'],
    [lr_pred, rf_pred], [C_LR, C_RF],
    [lr_m, rf_m]
):
    style(ax)
    ax.plot(dates_test, y_test, color=C_ACT, lw=2,   label='Actual Close')
    ax.plot(dates_test, pred,   color=col,   lw=1.5, label=f'{title} Predicted',
            linestyle='--', alpha=0.9)
    ax.fill_between(dates_test, y_test, pred, alpha=0.10, color=col)
    ax.set_title(f'{title}  |  RMSE=${m[0]:.2f}  MAE=${m[1]:.2f}  R²={m[2]:.4f}',
                 color=TEXT, fontsize=11, pad=7)
    ax.set_ylabel('Price (USD)', color=MUTED, fontsize=10)
    ax.legend(frameon=False, labelcolor=TEXT, fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

axes[-1].set_xlabel('Date', color=MUTED, fontsize=10)
plt.tight_layout()
plt.show()

### Plot 2 — Full Price History with Train/Test Split

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
fig.patch.set_facecolor(BG); style(ax)
ax.plot(dates_train, y_train, color=MUTED, lw=1.2, label='Training Data (80%)', alpha=0.6)
ax.plot(dates_test,  y_test,  color=C_ACT, lw=2,   label='Actual Close (Test 20%)')
ax.plot(dates_test,  lr_pred, color=C_LR,  lw=1.5, label='LR Predicted',  linestyle='--', alpha=0.85)
ax.plot(dates_test,  rf_pred, color=C_RF,  lw=1.5, label='RF Predicted',  linestyle=':',  alpha=0.85)
ax.axvline(dates_test[0], color='white', lw=1, linestyle=':', alpha=0.5)
ax.set_title('AAPL — Full Price History | Training vs Test Period',
             color=TEXT, fontsize=14, fontweight='bold')
ax.set_xlabel('Date', color=MUTED); ax.set_ylabel('Close Price (USD)', color=MUTED)
ax.legend(frameon=False, labelcolor=TEXT, fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout(); plt.show()

### Plot 3 — Feature Importance & Residuals

In [ ]:
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
rf_resid = y_test - rf_pred
lr_resid = y_test - lr_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(BG)
fig.suptitle('Feature Importance & Residual Analysis', color=TEXT, fontsize=14, fontweight='bold')

# Feature importance
style(ax1)
top = fi.head(8).sort_values()
bar_colors = [C_RF if i>=5 else MUTED for i in range(len(top))]
bars = ax1.barh(range(len(top)), top.values, color=bar_colors, alpha=0.85, edgecolor='none')
ax1.set_yticks(range(len(top)))
ax1.set_yticklabels(top.index, color=TEXT, fontsize=10)
ax1.set_xlabel('Importance Score', color=MUTED)
ax1.set_title('Top Feature Importances (RF)', color=TEXT, fontsize=11, fontweight='bold')
for bar, val in zip(bars, top.values):
    ax1.text(val+0.003, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', color=TEXT, fontsize=8)

# Residuals
style(ax2)
ax2.scatter(rf_pred, rf_resid, color=C_RF, alpha=0.5, s=20, label='Random Forest',     edgecolors='none')
ax2.scatter(lr_pred, lr_resid, color=C_LR, alpha=0.4, s=20, label='Linear Regression', edgecolors='none')
ax2.axhline(0, color='white', lw=1.2, linestyle='--', alpha=0.5)
ax2.set_xlabel('Predicted Price (USD)', color=MUTED)
ax2.set_ylabel('Residual  (Actual − Predicted)', color=MUTED)
ax2.set_title('Residual Plot — Prediction Errors', color=TEXT, fontsize=11, fontweight='bold')
ax2.legend(frameon=False, labelcolor=TEXT, fontsize=10)
plt.tight_layout(); plt.show()

## Results & Key Insights

### Model Comparison

| Model | RMSE | MAE | R² |
|---|---|---|---|
| Linear Regression | ~$3.05 | ~$2.44 | ~0.797 |
| Random Forest | ~$3.14 | ~$2.52 | ~0.786 |

---

### Key Findings

1. **Linear Regression performed slightly better than Random Forest** on this dataset.
   Stock prices have a strong linear autocorrelation (today's price ≈ yesterday's price),
   which LR captures efficiently.

2. **Prev_Close is the most important feature** (~42% importance in RF), followed by
   Prev_Open (~31%) and Prev_High (~14%). This confirms stock price has strong
   day-over-day momentum.

3. **Both models achieved R² ≈ 0.79–0.80**, meaning they explain about 80% of the
   variance in test prices — reasonable for a purely technical (price-based) model.

4. **Residuals are centered around zero** with no clear pattern, suggesting the models
   are not systematically biased — errors are mostly random noise.

5. **Prediction is hardest during high-volatility periods** (visible as larger gaps
   in the actual vs predicted plots) — expected, since no news or sentiment data is used.

---

### Limitations & Real-World Notes

- This model uses **only technical price features** — no news, earnings, or macro data
- Short-term stock prices are inherently noisy; even good models have limited accuracy
- For production use, consider **LSTM/GRU neural networks** for time-series patterns,
  or add **sentiment features** from news/social media

---